[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/05_awq_benchmark.ipynb)

# 05 · AWQ 4-bit + T4 benchmark (fp16 vs 8-bit vs AWQ)

Quantises the merged model with AutoAWQ (4-bit GEMM, group 128, calibrated on *training* postings) and benchmarks fp16, bitsandbytes 8-bit and AWQ on the same evenly spaced test subset at batch size 1: latency, tokens/s, peak VRAM, size on disk and accuracy. AutoAWQ is archived upstream, so it gets its own pinned environment; if it fails to install on your Colab image, the AWQ row stays *pending* and the other rows still run.

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~45 min**. Cells after the setup are safe to re-run.

**Outputs:** `results/benchmarks/t4.json`, predictions + metrics for `qwen2.5-1.5b_merged_int8` and `qwen2.5-1.5b_awq` (pushed to `<HF_USERNAME>/jobx-artifacts`); AWQ model pushed to `<HF_USERNAME>/jobx-qwen2.5-1.5b-awq`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q autoawq==0.2.9 transformers==4.51.3 accelerate==1.10.1 huggingface_hub==0.35.3 datasets==4.1.1 bitsandbytes==0.48.1
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

In [ ]:
N_BENCH = 150
SPLIT = 'test_real' if os.path.exists('data/real/test_real.jsonl') else 'test_synth'
import json, gc, torch
from pathlib import Path
from huggingface_hub import snapshot_download
from jobx.data.build import read_jsonl
from jobx.eval.run import SPLIT_FILES, eval_subset, git_commit
from jobx.baselines.hf_runner import load_model
from jobx.baselines.runs import push_results
from jobx.quantize.gpu_bench import bench_transformers, dir_size_gb, awq_quantize
from jobx.hub import repo_id, push_folder
MERGED = snapshot_download(repo_id('merged'))
EX = eval_subset(read_jsonl(SPLIT_FILES[SPLIT]), N_BENCH)
META = {'hardware': 'T4', 'commit': git_commit(), 'split': SPLIT}
rows = []

In [ ]:
def bench(variant, approach, quant=None, path=MERGED, size=None):
    model, tok = load_model(path, quant=quant)
    row = bench_transformers(model, tok, EX, variant=variant, approach=approach, split=SPLIT, size_gb=size,
                             meta={**META, 'model': path})
    print(row); rows.append(row)
    del model; gc.collect(); torch.cuda.empty_cache()
fp16_gb = dir_size_gb(MERGED)
bench('fp16 (transformers)', 'qwen2.5-1.5b_merged_fp16_bench', size=fp16_gb)
bench('merged 8-bit (bitsandbytes)', 'qwen2.5-1.5b_merged_int8', quant='8bit', size=round(fp16_gb / 2, 3))

## AWQ

In [ ]:
try:
    calib = [e['text'] for e in read_jsonl('data/processed/train.jsonl')[:128]]
    awq_dir = awq_quantize(MERGED, 'outputs/awq', calib)
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer
    amodel = AutoAWQForCausalLM.from_quantized(str(awq_dir), fuse_layers=False)
    atok = AutoTokenizer.from_pretrained(str(awq_dir)); atok.padding_side = 'left'
    row = bench_transformers(amodel.model, atok, EX, variant='AWQ 4-bit', approach='qwen2.5-1.5b_awq',
                             split=SPLIT, size_gb=dir_size_gb(awq_dir), meta={**META, 'model': 'awq'})
    print(row); rows.append(row)
    print('pushed', push_folder(awq_dir, 'awq', commit_message='AWQ 4-bit'))
except Exception as e:
    print('AWQ step failed, row stays pending:', repr(e))

In [ ]:
Path('results/benchmarks').mkdir(parents=True, exist_ok=True)
Path('results/benchmarks/t4.json').write_text(json.dumps({'rows': rows}, indent=2))
files = [Path('results/benchmarks/t4.json').resolve()]
for a in ('qwen2.5-1.5b_merged_int8', 'qwen2.5-1.5b_awq', 'qwen2.5-1.5b_merged_fp16_bench'):
    for suffix in ('.jsonl', '.meta.json'):
        p = Path(f'results/predictions/{a}__{SPLIT}{suffix}')
        if p.exists(): files.append(p.resolve())
push_results(files)
rows